In [ ]:
from google.colab import files
files.upload()


Saving archive.zip to archive (1).zip


In [ ]:
!unzip archive.zip
!ls -lh

In [ ]:
import pandas as pd
df = pd.read_csv('bank_fraud.csv')
print(f'Shape: {df.shape}')
df.head(

)

In [ ]:
# Missing Values
print('n=== MISSING VALUES === n')
print(df.isnull().sum())


In [ ]:
# Distribusi fraud
print('\n=== DISTRIBUSI TARGET ===')
print(df['is_fraud'].value_counts())
print(df['is_fraud'].value_counts(normalize=True) * 100)

In [ ]:
import matplotlib.pyplot as plt

labels = ['Not Fraud', 'Fraud']
values = [944745, 55255]
colors = ['steelblue', 'tomato']

plt.figure(figsize=(6, 4))
plt.bar(labels, values, color=colors)
plt.title('Distribusi Fraud vs Not Fraud')
plt.ylabel('Jumlah Transaksi')
for i, v in enumerate(values):
    plt.text(i, v + 5000, f'{v:,}\n({v/1000000*100:.1f}%)', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print(df.dtypes)


In [ ]:
from sklearn.preprocessing import LabelEncoder

df_model = df.copy()  # selalu copy dulu, jangan utak-atik df asli!

# Drop kolom yang tidak berguna
cols_to_drop = ['transaction_id', 'customer_id', 'transaction_date',
                'transaction_time', 'fraud_type']
df_model = df_model.drop(columns=cols_to_drop)

# Encode kolom teks jadi angka
le = LabelEncoder()
categorical_cols = ['country', 'city', 'merchant_category',
                    'payment_method', 'device_type']

for col in categorical_cols:
    df_model[col] = le.fit_transform(df_model[col])

print(f'Shape setelah preprocessing: {df_model.shape}')
print(f'Kolom yang tersisa: {list(df_model.columns)}')
print('\nSample hasil encoding:')
print(df_model[categorical_cols].head())

In [ ]:
from sklearn.model_selection import train_test_split

# Pisahkan fitur (X) dan target (y)
X = df_model.drop(columns=['is_fraud'])
y = df_model['is_fraud']

print(f'Jumlah fitur (X): {X.shape[1]} kolom')
print(f'Kolom fitur: {list(X.columns)}')

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,  # biar hasilnya reproducible — kalau dirun ulang hasilnya sama
    stratify=y
)

print(f'\nTrain set: {X_train.shape[0]:,} rows')
print(f'Test set:  {X_test.shape[0]:,} rows')
print(f'\nProporsi fraud di train: {y_train.mean():.4f} ({y_train.mean()*100:.2f}%)')
print(f'Proporsi fraud di test:  {y_test.mean():.4f} ({y_test.mean()*100:.2f}%)')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

print('🚀 Mulai training Random Forest...')
print('Estimasi waktu: 5-15 menit untuk 800k rows')
print('Santai dulu, jangan tutup tab Colab-nya!\n')

start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,         # 100 pohon keputusan
    max_depth=10,             # maksimal kedalaman pohon
    class_weight='balanced',  # handle imbalanced dataset
    random_state=42,          # reproducible
    n_jobs=-1                 # pakai semua CPU core
)

rf_model.fit(X_train, y_train)

elapsed = time.time() - start
print(f'✅ Training selesai dalam {elapsed/60:.1f} menit!')


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

# Model prediksi data test
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Hitung metrik
acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print('=== HASIL EVALUASI MODEL ===')
print(f'Accuracy:  {acc*100:.2f}%')
print(f'ROC-AUC:   {roc_auc:.4f}')
print()
print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues',
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'])
plt.title('Confusion Matrix')
plt.ylabel('Actual (Jawaban Asli)')
plt.xlabel('Predicted (Tebakan Model)')
plt.tight_layout()
plt.show()

print(f'Beneran Not Fraud, diprediksi Not Fraud : {tn:,} ✅')
print(f'Beneran Not Fraud, diprediksi Fraud     : {fp:,} ❌ False Alarm')
print(f'Beneran Fraud, diprediksi Not Fraud     : {fn:,} ❌ Missed Fraud')
print(f'Beneran Fraud, diprediksi Fraud         : {tp:,} ✅ Tertangkap!')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Feature importance
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('=== TOP 10 FITUR PALING PENTING ===')
print(importances.head(10).to_string(index=False))

# Visualisasi
plt.figure(figsize=(10, 6))
top10 = importances.head(10)
plt.barh(top10['Feature'][::-1], top10['Importance'][::-1], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Top 10 Fitur yang Paling Mempengaruhi Deteksi Fraud')
plt.tight_layout()
plt.show()

In [ ]:
import joblib

# Simpan model ke file
joblib.dump(rf_model, 'fraud_model.pkl')

# Simpan juga nama kolom yang dipakai
# Ini penting supaya nanti kalau load model, kita tau kolom apa yang dibutuhkan
import json
with open('model_features.json', 'w') as f:
    json.dump(list(X.columns), f)

print('✅ Model tersimpan: fraud_model.pkl')
print('✅ Fitur tersimpan: model_features.json')
print(f'Jumlah fitur: {len(X.columns)}')